Expire Changed Rows from Day 3

In [0]:
-- Step 1: Expire rows that have changed in Bronze (Day 3 modifications)
UPDATE nestle_dev_silver.core.sales_transactions_scd2 AS target
SET dbt_valid_to = (
    SELECT CAST(TO_TIMESTAMP(MAX(src_b.modified_at)) AS DATE)
    FROM nestle_dev_bronze.sql_server.sales_transactions src_b
    WHERE TRY_CAST(REGEXP_EXTRACT(src_b.transaction_id, '(\\d+)
    dbt_is_current = FALSE
WHERE dbt_is_current = TRUE
  AND EXISTS (
    SELECT 1 FROM nestle_dev_bronze.sql_server.sales_transactions AS src
    WHERE src.transaction_id = target.transaction_id
      AND (src.amount <> target.amount 
           OR src.quantity <> target.quantity
           OR src.channel <> target.channel)
  );

SELECT COUNT(*) as expired_rows FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = FALSE;  , 1) AS INT)
          = TRY_CAST(REGEXP_EXTRACT(target.transaction_id, '(\\d+)
    dbt_is_current = FALSE
WHERE dbt_is_current = TRUE
  AND EXISTS (
    SELECT 1 FROM nestle_dev_bronze.sql_server.sales_transactions AS src
    WHERE src.transaction_id = target.transaction_id
      AND (src.amount <> target.amount 
           OR src.quantity <> target.quantity
           OR src.channel <> target.channel)
  );

SELECT COUNT(*) as expired_rows FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = FALSE;  , 1) AS INT)
  ),
    dbt_is_current = FALSE
WHERE dbt_is_current = TRUE
  AND EXISTS (
    SELECT 1 FROM nestle_dev_bronze.sql_server.sales_transactions AS src
    WHERE src.transaction_id = target.transaction_id
      AND (src.amount <> target.amount 
           OR src.quantity <> target.quantity
           OR src.channel <> target.channel)
  );

SELECT COUNT(*) as expired_rows FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = FALSE;  

 Insert New Current Rows from Day 3 Bronze

In [0]:
-- Step 2: Insert new current rows (Day 3 additions + updated rows)
INSERT INTO nestle_dev_silver.core.sales_transactions_scd2
SELECT 
  src.transaction_id,
  src.product_id,
  src.region,
  src.channel,
  src.customer_id,
  src.quantity,
  src.unit_price,
  src.amount,
  CAST(TO_TIMESTAMP(src.modified_at) AS DATE) as dbt_valid_from,
  CAST('2099-12-31' AS DATE) as dbt_valid_to,
  TRUE as dbt_is_current
FROM nestle_dev_bronze.sql_server.sales_transactions src
WHERE NOT EXISTS (
  SELECT 1 FROM nestle_dev_silver.core.sales_transactions_scd2 tgt
  WHERE tgt.transaction_id = src.transaction_id
    AND tgt.dbt_is_current = TRUE
);

SELECT COUNT(*) as new_current_rows FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = TRUE;

 Verify SCD2 History

In [0]:
SELECT 
  COUNT(*) as total_rows,
  COUNT(CASE WHEN dbt_is_current = TRUE THEN 1 END) as current_rows,
  COUNT(CASE WHEN dbt_is_current = FALSE THEN 1 END) as expired_rows
FROM nestle_dev_silver.core.sales_transactions_scd2;